# 37 — External Bioactivity Data Fetch

Fetch PXR and related nuclear receptor bioactivity data from multiple public sources:
- PubChem BioAssay (PXR-specific assay AIDs)
- Tox21 NR pathway data (via DeepChem)
- BindingDB NR binding measurements (IC50/Ki)
- Extended ChEMBL NR targets (CAR + additional PXR measurement types)

All outputs cached to `data/external/`. Designed to be idempotent — if the cache file
already exists, load it instead of re-fetching.

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

import sys, warnings, time, math
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd

from pxr.data import load_train
from pxr.chem import standardize_smiles, to_inchikey
from pxr.external import (
    fetch_chembl_target, fetch_all_nr_targets,
    NUCLEAR_RECEPTOR_TARGETS, _get_json,
)
from pxr.paths import DATA_EXTERNAL, DATA_PROCESSED

DATA_EXTERNAL.mkdir(parents=True, exist_ok=True)

# Pre-compute training InChIKeys for overlap checks
train = load_train()
train_inchikeys = set(train['smiles'].map(to_inchikey).dropna())
print(f'Training set: {len(train):,} compounds  |  {len(train_inchikeys):,} unique InChIKeys')
print('Setup complete.')

Training set: 4,139 compounds  |  4,138 unique InChIKeys
Setup complete.


## 1. PubChem BioAssay PXR Data

Fetch actives and inactives from four PXR-specific BioAssay AIDs:
- AID 743219: PXR reporter assay
- AID 651631: PXR confirmatory assay
- AID 1224832: PXR qHTS
- AID 624202: PXR agonist assay

Active compounds receive pseudo-pEC50 = 6.0 (midpoint of hit range);
inactive compounds receive pseudo-pEC50 = 3.5.

In [2]:
# ── 1. PubChem BioAssay PXR data ──────────────────────────────────────────────
PUBCHEM_CACHE = DATA_EXTERNAL / 'pubchem_pxr_aids.parquet'
PUBCHEM_BASE  = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
PXR_AIDS = [743219, 651631, 1224832, 624202]

# Pseudo-pEC50 assignments: active = 6.0, inactive = 3.5
ACTIVE_PVAL   = 6.0
INACTIVE_PVAL = 3.5
MAX_CIDS      = 10_000   # per AID per activity class
SMILES_BATCH  = 100      # CIDs per SMILES fetch request
REQUEST_SLEEP = 0.3      # seconds between requests

if PUBCHEM_CACHE.exists():
    print(f'Loading cached PubChem data from {PUBCHEM_CACHE}')
    pubchem_df = pd.read_parquet(PUBCHEM_CACHE)
    print(f'  {len(pubchem_df):,} rows loaded')
else:
    all_rows = []

    for aid in PXR_AIDS:
        print(f'\nFetching AID {aid}...')
        for cids_type, pval_assigned in [('active', ACTIVE_PVAL), ('inactive', INACTIVE_PVAL)]:
            url = (
                f'{PUBCHEM_BASE}/assay/aid/{aid}/cids/JSON'
                f'?cids_type={cids_type}&list_return=listkey'
            )
            # Use direct CID fetch instead of listkey for simplicity
            direct_url = (
                f'{PUBCHEM_BASE}/assay/aid/{aid}/cids/JSON'
                f'?cids_type={cids_type}'
            )
            try:
                resp = requests.get(direct_url, timeout=60)
                resp.raise_for_status()
                data = resp.json()
                time.sleep(REQUEST_SLEEP)
            except Exception as e:
                print(f'  AID {aid} {cids_type}: fetch failed — {e}')
                continue

            # Extract CID list
            inner = data.get('InformationList', {}).get('Information', [])
            if not inner:
                # Try alternative JSON structure
                cids_raw = data.get('IdentifierList', {}).get('CID', [])
            else:
                cids_raw = []
                for item in inner:
                    cids_raw.extend(item.get('CID', []))

            cids = [int(c) for c in cids_raw if c][:MAX_CIDS]
            print(f'  AID {aid} {cids_type}: {len(cids):,} CIDs')
            if not cids:
                continue

            # Fetch SMILES in batches
            cid_to_smi = {}
            for i in range(0, len(cids), SMILES_BATCH):
                batch_cids = cids[i:i + SMILES_BATCH]
                cids_csv = ','.join(map(str, batch_cids))
                smi_url = (
                    f'{PUBCHEM_BASE}/compound/cid/{cids_csv}'
                    f'/property/IsomericSMILES/JSON'
                )
                try:
                    smi_data = _get_json(smi_url)
                    if smi_data:
                        for prop in smi_data.get('PropertyTable', {}).get('Properties', []):
                            cid_to_smi[int(prop['CID'])] = prop.get('IsomericSMILES', '')
                except Exception as e:
                    print(f'    SMILES batch {i//SMILES_BATCH}: {e}')
                time.sleep(REQUEST_SLEEP)

            print(f'  AID {aid} {cids_type}: {len(cid_to_smi):,} SMILES resolved')
            for cid, smi in cid_to_smi.items():
                if smi:
                    all_rows.append({
                        'cid':      cid,
                        'smiles':   smi,
                        'aid':      aid,
                        'activity': cids_type,
                        'pec50':    pval_assigned,
                    })

    if all_rows:
        pubchem_df = pd.DataFrame(all_rows)
        # Standardize SMILES
        print('\nStandardizing SMILES...')
        pubchem_df['std_smiles'] = pubchem_df['smiles'].map(standardize_smiles)
        pubchem_df['inchikey']   = pubchem_df['std_smiles'].map(
            lambda s: to_inchikey(s) if s else None
        )
        pubchem_df = pubchem_df.dropna(subset=['std_smiles', 'inchikey']).reset_index(drop=True)
        # For duplicates (same compound in multiple AIDs), take the higher pEC50
        # (active wins over inactive assignment)
        pubchem_df = (
            pubchem_df.sort_values('pec50', ascending=False)
            .drop_duplicates(subset='inchikey', keep='first')
            .reset_index(drop=True)
        )
        pubchem_df.to_parquet(PUBCHEM_CACHE, index=False)
        print(f'Saved {len(pubchem_df):,} unique compounds to {PUBCHEM_CACHE}')
    else:
        print('No PubChem data retrieved — creating empty placeholder')
        pubchem_df = pd.DataFrame(
            columns=['cid', 'smiles', 'std_smiles', 'inchikey', 'aid', 'activity', 'pec50']
        )
        pubchem_df.to_parquet(PUBCHEM_CACHE, index=False)

overlap_pc = len(set(pubchem_df['inchikey'].dropna()) & train_inchikeys)
print(f'\nPubChem summary:')
print(f'  Total unique compounds: {len(pubchem_df):,}')
print(f'  Overlap with PXR train: {overlap_pc:,}')
if len(pubchem_df) > 0 and 'activity' in pubchem_df.columns:
    print(pubchem_df['activity'].value_counts().to_string())


Fetching AID 743219...


  AID 743219 active: 996 CIDs


  AID 743219 active: 996 SMILES resolved


  AID 743219 inactive: 5,131 CIDs


  AID 743219 inactive: 5,131 SMILES resolved

Fetching AID 651631...


...

  AID 1224832 inactive: fetch failed — 404 Client Error: PUGREST.NotFound for url: https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/1224832/cids/JSON?cids_type=inactive

Fetching AID 624202...


  AID 624202 active: 3,978 CIDs


  AID 624202 active: 3,978 SMILES resolved


  AID 624202 inactive: 10,000 CIDs


  AID 624202 inactive: 10,000 SMILES resolved
No PubChem data retrieved — creating empty placeholder

PubChem summary:
  Total unique compounds: 0
  Overlap with PXR train: 0


## 2. Tox21 NR Pathway Data

Load the Tox21 dataset via DeepChem and extract nuclear receptor (NR) pathway tasks.
These include AhR, AR, ER, and PPAR-gamma endpoints — structurally related to PXR binding.

In [3]:
# ── 2. Tox21 NR pathway data ──────────────────────────────────────────────────
TOX21_CACHE = DATA_EXTERNAL / 'tox21_nr_data.parquet'
NR_TASKS = ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma']

if TOX21_CACHE.exists():
    print(f'Loading cached Tox21 data from {TOX21_CACHE}')
    tox21_df = pd.read_parquet(TOX21_CACHE)
    print(f'  {len(tox21_df):,} rows loaded')
else:
    try:
        import deepchem as dc
        print(f'DeepChem version: {dc.__version__}')
        print('Loading Tox21 dataset...')
        tasks, datasets, transformers = dc.molnet.load_tox21()
        train_ds, valid_ds, test_ds = datasets

        # Gather all splits
        all_smiles, all_labels = [], []
        for ds in [train_ds, valid_ds, test_ds]:
            ids  = ds.ids          # array of SMILES strings
            labs = ds.y            # (N, n_tasks)
            all_smiles.extend(ids)
            all_labels.append(labs)

        all_labels = np.vstack(all_labels)
        full_tasks = list(tasks)
        print(f'Tox21 tasks: {full_tasks}')
        print(f'Total compounds: {len(all_smiles):,}')

        rows = []
        for task in NR_TASKS:
            if task not in full_tasks:
                print(f'  Task {task} not found in Tox21 — skipping')
                continue
            task_idx = full_tasks.index(task)
            for i, smi in enumerate(all_smiles):
                lab = all_labels[i, task_idx]
                # NaN / -1 means not tested
                if np.isnan(lab) or lab < 0:
                    continue
                rows.append({
                    'smiles':       smi,
                    'task_name':    task,
                    'binary_label': int(lab),
                })

        tox21_df = pd.DataFrame(rows)
        print(f'Extracted {len(tox21_df):,} NR pathway records')

        # Standardize
        tox21_df['std_smiles'] = tox21_df['smiles'].map(standardize_smiles)
        tox21_df['inchikey']   = tox21_df['std_smiles'].map(
            lambda s: to_inchikey(s) if s else None
        )
        tox21_df = tox21_df.dropna(subset=['std_smiles']).reset_index(drop=True)
        tox21_df.to_parquet(TOX21_CACHE, index=False)
        print(f'Saved to {TOX21_CACHE}')

    except ImportError:
        print('DeepChem not available — creating empty placeholder')
        tox21_df = pd.DataFrame(
            columns=['smiles', 'std_smiles', 'inchikey', 'task_name', 'binary_label']
        )
        tox21_df.to_parquet(TOX21_CACHE, index=False)
    except Exception as e:
        print(f'Tox21 fetch failed: {e}')
        tox21_df = pd.DataFrame(
            columns=['smiles', 'std_smiles', 'inchikey', 'task_name', 'binary_label']
        )
        tox21_df.to_parquet(TOX21_CACHE, index=False)

print(f'\nTox21 summary:')
print(f'  Total records:     {len(tox21_df):,}')
if len(tox21_df) > 0 and 'task_name' in tox21_df.columns:
    print(tox21_df.groupby('task_name')['binary_label'].value_counts().to_string())

DeepChem not available — creating empty placeholder

Tox21 summary:
  Total records:     0


## 3. BindingDB NR Binding Data

Fetch IC50/Ki data from BindingDB REST API for eight nuclear receptor UniProt IDs.
Falls back to ChEMBL `chembl_webresource_client` if the BindingDB API is unavailable.
Converts IC50 (nM) to pIC50 = -log10(IC50 * 1e-9).

In [4]:
# ── 3. BindingDB NR binding data ──────────────────────────────────────────────
BINDINGDB_CACHE = DATA_EXTERNAL / 'bindingdb_nr_data.parquet'

NR_UNIPROTS = {
    'PXR':  'O75469',
    'VDR':  'P11473',
    'FXR':  'Q96RI1',
    'LXRa': 'Q13133',
    'RXRa': 'P19793',
    'PPARg':'P37231',
    'PPARa':'Q07869',
    'CAR':  'O9U8V6',
}

BDB_URL_TEMPLATE = (
    'https://bindingdb.org/axis2/services/BDBService/getLigandsByUniprots'
    '?uniprot={uid}&response=json'
)

# IC50 filter range (nM)
IC50_MIN_NM = 0.01
IC50_MAX_NM = 100_000

def ic50_nm_to_pic50(ic50_nm: float) -> float | None:
    """Convert IC50 in nM to pIC50 (-log10 molar)."""
    if ic50_nm <= 0 or not np.isfinite(ic50_nm):
        return None
    return -math.log10(ic50_nm * 1e-9)


def fetch_bindingdb_target(uniprot: str, target_name: str) -> pd.DataFrame:
    """Fetch IC50/Ki records from BindingDB for one UniProt ID."""
    url = BDB_URL_TEMPLATE.format(uid=uniprot)
    try:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f'  BindingDB {target_name} ({uniprot}): request failed — {e}')
        return pd.DataFrame()

    affinity_records = data.get('affinities', data.get('getLigandsByUniprotsResponse', {}).get('affinities', []))
    if isinstance(affinity_records, dict):
        affinity_records = affinity_records.get('affinity', [])
    if not isinstance(affinity_records, list):
        affinity_records = [affinity_records]

    rows = []
    for rec in affinity_records:
        if not isinstance(rec, dict):
            continue
        smiles = rec.get('smiles', '') or rec.get('ligand_smiles', '')
        if not smiles:
            continue
        # Extract IC50/Ki value
        affinity_type = str(rec.get('affinity_type', '') or '').strip().upper()
        if affinity_type not in ('IC50', 'KI', 'EC50', 'KD'):
            continue
        try:
            val_nm = float(rec.get('affinity', rec.get('affinity_value', 'nan')))
        except (ValueError, TypeError):
            continue
        if not (IC50_MIN_NM <= val_nm <= IC50_MAX_NM):
            continue
        pic50 = ic50_nm_to_pic50(val_nm)
        if pic50 is None:
            continue
        rows.append({
            'smiles':          smiles,
            'pec50':           pic50,
            'affinity_nm':     val_nm,
            'affinity_type':   affinity_type,
            'target_name':     target_name,
            'uniprot':         uniprot,
        })

    return pd.DataFrame(rows)


if BINDINGDB_CACHE.exists():
    print(f'Loading cached BindingDB data from {BINDINGDB_CACHE}')
    bdb_df = pd.read_parquet(BINDINGDB_CACHE)
    print(f'  {len(bdb_df):,} rows loaded')
else:
    bdb_rows = []
    bindingdb_available = True

    for target_name, uniprot in NR_UNIPROTS.items():
        print(f'Fetching BindingDB: {target_name} ({uniprot})...')
        df_t = fetch_bindingdb_target(uniprot, target_name)
        if not df_t.empty:
            bdb_rows.append(df_t)
            print(f'  {target_name}: {len(df_t):,} records')
        else:
            print(f'  {target_name}: no records')
            bindingdb_available = False
        time.sleep(REQUEST_SLEEP)

    if bdb_rows:
        bdb_df = pd.concat(bdb_rows, ignore_index=True)
        bdb_df['std_smiles'] = bdb_df['smiles'].map(standardize_smiles)
        bdb_df['inchikey']   = bdb_df['std_smiles'].map(
            lambda s: to_inchikey(s) if s else None
        )
        bdb_df = bdb_df.dropna(subset=['std_smiles', 'inchikey']).reset_index(drop=True)
        bdb_df.to_parquet(BINDINGDB_CACHE, index=False)
        print(f'\nSaved {len(bdb_df):,} BindingDB records to {BINDINGDB_CACHE}')
    else:
        # Fallback: ChEMBL chembl_webresource_client
        print('BindingDB unavailable — falling back to ChEMBL webresource client...')
        chembl_rows = []
        try:
            from chembl_webresource_client.new_client import new_client
            activity_client = new_client.activity
            for name, chembl_id in NUCLEAR_RECEPTOR_TARGETS.items():
                print(f'  ChEMBL fallback: {name} ({chembl_id})')
                try:
                    acts = activity_client.filter(
                        target_chembl_id=chembl_id,
                        standard_type__in=['IC50', 'Ki'],
                        pchembl_value__isnull=False,
                    ).only(['canonical_smiles', 'pchembl_value', 'standard_type'])
                    for act in acts:
                        smi = act.get('canonical_smiles', '')
                        pval = act.get('pchembl_value')
                        if smi and pval:
                            try:
                                chembl_rows.append({
                                    'smiles':        smi,
                                    'pec50':         float(pval),
                                    'affinity_type': act.get('standard_type', ''),
                                    'target_name':   name,
                                    'uniprot':       NR_UNIPROTS.get(name, ''),
                                })
                            except (ValueError, TypeError):
                                pass
                except Exception as e:
                    print(f'    ChEMBL fallback {name}: {e}')
        except ImportError:
            print('  chembl_webresource_client not available either')

        if chembl_rows:
            bdb_df = pd.DataFrame(chembl_rows)
            bdb_df['std_smiles'] = bdb_df['smiles'].map(standardize_smiles)
            bdb_df['inchikey']   = bdb_df['std_smiles'].map(
                lambda s: to_inchikey(s) if s else None
            )
            bdb_df = bdb_df.dropna(subset=['std_smiles', 'inchikey']).reset_index(drop=True)
        else:
            bdb_df = pd.DataFrame(
                columns=['smiles', 'std_smiles', 'inchikey', 'pec50',
                         'affinity_type', 'target_name', 'uniprot']
            )
        bdb_df.to_parquet(BINDINGDB_CACHE, index=False)
        print(f'Saved {len(bdb_df):,} records (via fallback) to {BINDINGDB_CACHE}')

overlap_bdb = len(set(bdb_df['inchikey'].dropna()) & train_inchikeys) if len(bdb_df) > 0 else 0
print(f'\nBindingDB summary:')
print(f'  Total records:          {len(bdb_df):,}')
print(f'  Overlap with PXR train: {overlap_bdb:,}')
if len(bdb_df) > 0 and 'target_name' in bdb_df.columns:
    print(bdb_df.groupby('target_name')['pec50'].describe().round(2).to_string())

Fetching BindingDB: PXR (O75469)...


  BindingDB PXR (O75469): request failed — 404 Client Error: Not Found for url: https://bindingdb.org/axis2/services/BDBService/getLigandsByUniprots?uniprot=O75469&response=json
  PXR: no records


Fetching BindingDB: VDR (P11473)...


  BindingDB VDR (P11473): request failed — 404 Client Error: Not Found for url: https://bindingdb.org/axis2/services/BDBService/getLigandsByUniprots?uniprot=P11473&response=json
  VDR: no records


Fetching BindingDB: FXR (Q96RI1)...


...

  ChEMBL fallback: LXRa (CHEMBL2808)


  ChEMBL fallback: PPARg (CHEMBL235)


  ChEMBL fallback: PPARa (CHEMBL2111325)


  ChEMBL fallback: RXRa (CHEMBL2061)


Saved 5,690 records (via fallback) to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\external\bindingdb_nr_data.parquet

BindingDB summary:
  Total records:          5,690
  Overlap with PXR train: 13
              count  mean   std   min   25%   50%   75%    max
target_name                                                   
FXR           456.0  5.95  1.38  4.00  5.01  5.48  6.68  10.30
LXRa          856.0  6.58  1.08  4.04  5.77  6.62  7.30   9.10
PPARg        2493.0  6.65  1.18  4.00  5.70  6.59  7.55  10.30
PXR           346.0  6.21  0.90  4.39  5.60  6.13  6.68   9.00
RXRa         1061.0  6.97  1.04  4.08  6.14  6.99  7.82   9.40
VDR           478.0  6.45  1.60  4.08  4.98  6.14  8.04  10.54


## 4. Extended ChEMBL NR Targets

Supplement the existing ChEMBL NR target data with:
- CAR target (CHEMBL3509594) if not already present
- Additional PXR measurement types: ADMET, EC50, Ki (not just IC50)
- Merged and deduplicated into `chembl_nr_extended.parquet`

In [5]:
# ── 4. Extended ChEMBL NR targets ─────────────────────────────────────────────
CHEMBL_BASE_CACHE    = DATA_EXTERNAL / 'chembl_nr_targets.parquet'
CHEMBL_EXT_CACHE     = DATA_EXTERNAL / 'chembl_nr_extended.parquet'

# Full target registry including CAR
EXTENDED_TARGETS = dict(NUCLEAR_RECEPTOR_TARGETS)   # copy from external.py
EXTENDED_TARGETS['CAR'] = 'CHEMBL3509594'           # already in NUCLEAR_RECEPTOR_TARGETS but make explicit

# Measurement types to fetch — broader than the default IC50-only set
EXTENDED_TYPES = ('IC50', 'EC50', 'Ki', 'Kd', 'AC50', 'GI50', 'potency')

if CHEMBL_EXT_CACHE.exists():
    print(f'Loading cached extended ChEMBL data from {CHEMBL_EXT_CACHE}')
    chembl_ext_df = pd.read_parquet(CHEMBL_EXT_CACHE)
    print(f'  {len(chembl_ext_df):,} rows loaded')
else:
    # Load existing base cache if available
    if CHEMBL_BASE_CACHE.exists():
        print(f'Loading existing base cache: {CHEMBL_BASE_CACHE}')
        base_df = pd.read_parquet(CHEMBL_BASE_CACHE)
        existing_targets = set(base_df['chembl_id'].unique()) if 'chembl_id' in base_df.columns else set()
        print(f'  {len(base_df):,} existing records, targets: {existing_targets}')
    else:
        base_df = pd.DataFrame()
        existing_targets = set()
        print('No base ChEMBL cache found — fetching all targets fresh')

    # Fetch missing targets or supplement PXR with extended measurement types
    new_dfs = [base_df] if len(base_df) > 0 else []

    for target_name, chembl_id in EXTENDED_TARGETS.items():
        should_fetch = (
            chembl_id not in existing_targets
            or target_name == 'PXR'  # always re-fetch PXR with broader types
        )
        if not should_fetch:
            print(f'  {target_name} ({chembl_id}): already in cache — skipping')
            continue

        print(f'  Fetching {target_name} ({chembl_id}) with extended types...')
        try:
            df_t = fetch_chembl_target(
                chembl_id,
                standard_types=EXTENDED_TYPES,
                verbose=True,
            )
            if not df_t.empty:
                df_t['target_name'] = target_name
                new_dfs.append(df_t)
        except Exception as e:
            print(f'    {target_name}: fetch failed — {e}')
        time.sleep(0.5)

    if new_dfs:
        chembl_ext_df = pd.concat(new_dfs, ignore_index=True)

        # Standardize SMILES and deduplicate per target
        chembl_ext_df['std_smiles'] = chembl_ext_df['smiles'].map(standardize_smiles)
        chembl_ext_df['inchikey']   = chembl_ext_df['std_smiles'].map(
            lambda s: to_inchikey(s) if s else None
        )
        chembl_ext_df = chembl_ext_df.dropna(subset=['std_smiles', 'inchikey'])

        # For rows with same inchikey + target_name, keep median pec50
        key_cols = ['inchikey', 'target_name'] if 'target_name' in chembl_ext_df.columns else ['inchikey']
        chembl_ext_df = (
            chembl_ext_df.groupby(key_cols, as_index=False)
            .agg(
                smiles      =('smiles', 'first'),
                std_smiles  =('std_smiles', 'first'),
                pec50       =('pec50', 'median'),
                standard_type=('standard_type', 'first'),
                chembl_id   =('chembl_id', 'first'),
            )
            .reset_index(drop=True)
        )

        chembl_ext_df.to_parquet(CHEMBL_EXT_CACHE, index=False)
        print(f'\nSaved {len(chembl_ext_df):,} extended ChEMBL records to {CHEMBL_EXT_CACHE}')
    else:
        print('No ChEMBL data available')
        chembl_ext_df = pd.DataFrame(
            columns=['smiles', 'std_smiles', 'inchikey', 'pec50',
                     'standard_type', 'chembl_id', 'target_name']
        )
        chembl_ext_df.to_parquet(CHEMBL_EXT_CACHE, index=False)

overlap_chembl = len(set(chembl_ext_df['inchikey'].dropna()) & train_inchikeys) if len(chembl_ext_df) > 0 else 0
print(f'\nChEMBL extended summary:')
print(f'  Total records:          {len(chembl_ext_df):,}')
print(f'  Overlap with PXR train: {overlap_chembl:,}')
if len(chembl_ext_df) > 0 and 'target_name' in chembl_ext_df.columns:
    print(chembl_ext_df.groupby('target_name')['pec50'].describe().round(2).to_string())

Loading existing base cache: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\external\chembl_nr_targets.parquet
  11,511 existing records, targets: {'CHEMBL2808', 'CHEMBL2111325', 'CHEMBL3401', 'CHEMBL2061', 'CHEMBL1977', 'CHEMBL235', 'CHEMBL2047'}
  Fetching PXR (CHEMBL3401) with extended types...


  CHEMBL3401: 947 unique SMILES  (pec50 4.00–8.62)


  Fetching CAR (CHEMBL3509594) with extended types...


  CHEMBL3509594: 0 records found


  VDR (CHEMBL1977): already in cache — skipping
  FXR (CHEMBL2047): already in cache — skipping
  LXRa (CHEMBL2808): already in cache — skipping
  PPARg (CHEMBL235): already in cache — skipping
  PPARa (CHEMBL2111325): already in cache — skipping
  RXRa (CHEMBL2061): already in cache — skipping



Saved 11,496 extended ChEMBL records to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\external\chembl_nr_extended.parquet

ChEMBL extended summary:
  Total records:          11,496
  Overlap with PXR train: 58
              count   mean   std    min    25%    50%    75%    max
target_name                                                        
FXR          3185.0   6.60  1.07   4.01   5.87   6.55   7.26  10.05
LXRa         1173.0   6.29  0.97   4.04   5.61   6.24   6.89   9.10
PPARa           4.0  10.71  0.24  10.35  10.66  10.80  10.84  10.89
PPARg        4302.0   6.38  1.11   4.00   5.52   6.17   7.15  10.74
PXR           945.0   5.60  0.83   4.00   4.99   5.52   6.10   8.62
RXRa         1364.0   6.70  1.09   4.08   5.89   6.73   7.58   9.40
VDR           523.0   6.77  1.53   4.17   5.33   6.85   8.07  11.00


## 5. Summary Statistics

Print row counts, pEC50/pIC50 distributions per source, total unique SMILES count,
and overlap with the PXR training set.

In [6]:
# ── 5. Summary statistics ──────────────────────────────────────────────────────
sources = {
    'PubChem PXR AIDs':     pubchem_df,
    'Tox21 NR':             tox21_df,
    'BindingDB NR':         bdb_df,
    'ChEMBL NR extended':   chembl_ext_df,
}

print('=' * 70)
print(f'{"Source":<25} {"Rows":>8} {"pEC50 mean":>12} {"pEC50 std":>10} {"Train overlap":>14}')
print('-' * 70)

all_inchikeys = set()
for src_name, df in sources.items():
    n = len(df)
    if n == 0:
        print(f'{src_name:<25} {n:>8}   (empty)')
        continue

    # pec50 column may not exist in tox21 (binary labels)
    if 'pec50' in df.columns:
        mean_p = df['pec50'].mean()
        std_p  = df['pec50'].std()
        pval_str = f'{mean_p:>12.3f} {std_p:>10.3f}'
    elif 'binary_label' in df.columns:
        pval_str = f'  binary={df["binary_label"].mean():.2f}     N/A'
    else:
        pval_str = f'         N/A        N/A'

    if 'inchikey' in df.columns:
        ik_set = set(df['inchikey'].dropna())
        all_inchikeys |= ik_set
        overlap = len(ik_set & train_inchikeys)
    else:
        overlap = 0

    print(f'{src_name:<25} {n:>8} {pval_str} {overlap:>14,}')

print('=' * 70)
print(f'\nTotal unique InChIKeys across all external sources: {len(all_inchikeys):,}')
print(f'Total overlap with PXR training set: {len(all_inchikeys & train_inchikeys):,}')
print(f'Novel compounds (not in training):   {len(all_inchikeys - train_inchikeys):,}')

print('\nCache files written:')
for path in [
    DATA_EXTERNAL / 'pubchem_pxr_aids.parquet',
    DATA_EXTERNAL / 'tox21_nr_data.parquet',
    DATA_EXTERNAL / 'bindingdb_nr_data.parquet',
    DATA_EXTERNAL / 'chembl_nr_extended.parquet',
]:
    exists = path.exists()
    size   = f'{path.stat().st_size / 1024:.1f} KB' if exists else 'missing'
    print(f'  {path.name:<35s}  {size}')

Source                        Rows   pEC50 mean  pEC50 std  Train overlap
----------------------------------------------------------------------
PubChem PXR AIDs                 0   (empty)
Tox21 NR                         0   (empty)
BindingDB NR                  5690        6.597      1.215             13
ChEMBL NR extended           11496        6.426      1.125             58

Total unique InChIKeys across all external sources: 11,243
Total overlap with PXR training set: 58
Novel compounds (not in training):   11,185

Cache files written:
  pubchem_pxr_aids.parquet             3.3 KB
  tox21_nr_data.parquet                2.7 KB
  bindingdb_nr_data.parquet            256.0 KB
  chembl_nr_extended.parquet           889.8 KB
